In [2]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
# If this shows system Python instead of tf_env, you need to select the tf_env_kernel kernel
# Click the kernel selector in the top right and choose "Python (TensorFlow Env)"

Python executable: c:\Python310\python.exe
Python version: 3.10.10 (tags/v3.10.10:aad5f6a, Feb  7 2023, 17:20:36) [MSC v.1929 64 bit (AMD64)]


In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier, 
                              StackingClassifier, BaggingClassifier, 
                              GradientBoostingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier 
import plotly
import plotly.graph_objs as go
import plotly.express as px
from plotly.offline import init_notebook_mode, plot, iplot
from sklearn.preprocessing import StandardScaler
import glob
import os
import tensorflow
from tensorflow.keras.preprocessing.image import load_img, img_to_array, array_to_img, ImageDataGenerator
from tensorflow.keras.layers import Conv2D, Flatten, MaxPooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.vgg16 import VGG16
from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm
from termcolor import colored

from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
print("Current tensorflow version == {}".format(tensorflow. __version__))

Current tensorflow version == 2.21.0


# No of images in each dataset

In [3]:
train_df = glob.glob(r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\train\**\*.jpeg", recursive=True)
test_df = glob.glob(r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\test\**\*.jpeg", recursive=True)
validation_df = glob.glob(r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\val\**\*.jpeg", recursive=True)


In [4]:
print("There is {} images in the training dataset".format(len(train_df)))
print("There is {} images in the test dataset".format(len(test_df)))
print("There is {} images in the validation dataset".format(len(validation_df)))

There is 5216 images in the training dataset
There is 624 images in the test dataset
There is 16 images in the validation dataset


# No of the pictures are of pneumonic lungs and No of them are of normal lungs

In [ ]:
datasets, pneumonia_lung, normal_lung = ["train", "test", "val"], [], []
import os
directory = r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray"
for i in datasets:
    path = os.path.join(directory, i)
    normal = glob.glob(os.path.join(path, "NORMAL/*.jpeg"))
    pneumonia = glob.glob(os.path.join(path, "PNEUMONIA/*.jpeg"))
    normal_lung.extend(normal), pneumonia_lung.extend(pneumonia)

print("The number of pneumonia images is {}".format(len(pneumonia_lung)))
print("The number of non-pneumonia images is {}".format(len(normal_lung)))

#  Shuffle the images 

In [ ]:
import random
random.shuffle(normal_lung)
random.shuffle(pneumonia_lung)
images = normal_lung[:50] + pneumonia_lung[:50]
images[:10]

# View the images in X-ray format

In [ ]:
normal_lung_image = load_img(r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\test\NORMAL\NORMAL2-IM-0370-0001.jpeg")
print("NORMAL")
plt.imshow(normal_lung_image)
plt.show()

In [ ]:
normal_lung_image = load_img(r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\train\PNEUMONIA\person409_virus_816.jpeg")
print("PNEUMONIA")
plt.imshow(normal_lung_image)
plt.show()

In [ ]:
!pip install opencv-python

In [ ]:
import cv2
fig = plt.figure(figsize = (20, 15))
columns, rows = 3, 3
for i in range(1, 10):
    img = cv2.imread(images[i])
    img = cv2.resize(img, (512, 512))
    fig.add_subplot(rows, columns, i)
    plt.imshow(img)

# Image erosion

In [ ]:
fig = plt.figure(figsize = (20, 15))
columns, rows = 3, 3
for i in range(1, 10):
    img = cv2.imread(images[i])
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    kernel = np.ones((5, 5), np.uint8)
    image_erosion = cv2.erode(img, kernel, iterations=3)
    fig.add_subplot(rows, columns, i)
    plt.imshow(image_erosion)

# Image dilation

In [ ]:
fig = plt.figure(figsize = (20, 15))
columns, rows = 3, 3

for i in range(1, 10):
    img = cv2.imread(images[i])
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    kernel = np.ones((5, 5), np.uint8)
    image_dilation = cv2.dilate(img, kernel, iterations = 2)
    fig.add_subplot(rows, columns, i)
    plt.imshow(image_dilation)

# Convert the images to greyscale and then apply Gaussian blur to them

In [ ]:
fig = plt.figure(figsize = (20, 15))
columns, rows = 3, 3

for i in range(1, 10):
    img = cv2.imread(images[i])
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img = cv2.addWeighted (img, 4, cv2.GaussianBlur(img, (0, 0), 512/10), -4, 128)
    fig.add_subplot(rows, columns, i)
    plt.imshow(img)
    plt.axis(False)

# Canny edge detection

In [ ]:
fig = plt.figure(figsize = (20, 15))
columns, rows = 3, 3

for i in range(1, 10):
    img = cv2.imread(images[i])
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    detected_edges = cv2.Canny(img, 80, 100)
    fig.add_subplot(rows, columns, i)
    plt.imshow(detected_edges)

In [ ]:
train_dir = r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\train"
test_dir = r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\test"
validation_dir = r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\val"

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model  # Import Model class
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier

In [ ]:
# Load pre-trained VGG16 model + higher level layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [ ]:
# Add custom layers on top of VGG16
model = Model(inputs=base_model.input, outputs=base_model.output)

In [ ]:
# Load and preprocess images
def load_images(image_paths, target_size=(128, 128)):
    images = []
    labels = []
    
    for img_path in image_paths:
        img = load_img(img_path, target_size=target_size)
        img = img_to_array(img)
        images.append(img)
        label = 1 if 'PNEUMONIA' in img_path else 0
        labels.append(label)
    
    return np.array(images), np.array(labels)

In [ ]:
# Define directories
train_dir = r"C:\Users\Parkavi R\Downloads\Project\chest_xray\chest_xray\train"
normal_images = glob.glob(os.path.join(train_dir, "NORMAL/*.jpeg"))
pneumonia_images = glob.glob(os.path.join(train_dir, "PNEUMONIA/*.jpeg"))

In [ ]:
# Load images and labels
image_paths = normal_images + pneumonia_images
X, y = load_images(image_paths)

In [ ]:
# Flatten images for machine learning algorithms
X_flat = X.reshape(X.shape[0], -1)

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize classifiers for ensemble methods
nb_classifier = GaussianNB()
knn_classifier = KNeighborsClassifier(n_neighbors=5)
dt_classifier = DecisionTreeClassifier(random_state=42)
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
bagging_classifier = BaggingClassifier(estimator=dt_classifier, n_estimators=10)
boosting_classifier = GradientBoostingClassifier(n_estimators=100)

In [ ]:
!pip install xgboost
!pip install lightgb

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# Add XGBoost, LightGBM, and KMeans
#xgb_classifier = XGBClassifier()
lgbm_classifier = LGBMClassifier()

In [ ]:
# Create a stacking classifier using the base classifiers
stacking_classifier = StackingClassifier(
    estimators=[
        ('nb', nb_classifier),
        ('knn', knn_classifier),
        ('dt', dt_classifier)
    ],
    final_estimator=RandomForestClassifier(n_estimators=100)
)

In [ ]:
# Train classifiers and store training history for loss plotting later
classifiers = {
    "Naive Bayes": nb_classifier,
    "KNN": knn_classifier,
    #"XGBoost": xgb_classifier,
    "LightGBM": lgbm_classifier,
    "Decision Tree": dt_classifier,
    "Random Forest": rf_classifier    
    #"Bagging": bagging_classifier,
    #"Boosting": boosting_classifier,
    #"Stacking": stacking_classifier
}

In [ ]:
training_losses = {}
validation_losses = {}

In [ ]:
for name, clf in classifiers.items():
    try:
        clf.fit(X_train_scaled, y_train)
        print(f"Successfully fitted {name}.")
    except Exception as e:
        print(f"Error fitting {name}: {e}")

In [ ]:
# Make predictions and print classification reports for each classifier
for name, clf in classifiers.items():
    predictions = clf.predict(X_test_scaled)
    print(f"{name} Classification Report:\n", classification_report(y_test, predictions))

In [ ]:
# Voting Classifier with all classifiers included
voting_classifier = VotingClassifier(
    estimators=[
        ('nb', nb_classifier),
        ('knn', knn_classifier),
        ('dt', dt_classifier),
        #('xgb', xgb_classifier),
        ('lgbm', lgbm_classifier),
        ('rf',rf_classifier)
        #('bagging', bagging_classifier),
       # ('boosting', boosting_classifier),
        #('stacking', stacking_classifier)
    ],
    voting='hard'
)

In [ ]:
voting_classifier.fit(X_train_scaled, y_train)
voting_predictions = voting_classifier.predict(X_test_scaled)

In [ ]:
print("Voting Classifier Classification Report:\n", classification_report(y_test, voting_predictions))

In [ ]:
# Confusion Matrices for all models in heatmap form
models_predictions = {name: clf.predict(X_test_scaled) for name, clf in classifiers.items()}
models_predictions['Voting Classifier'] = voting_predictions

In [ ]:
plt.figure(figsize=(12, 10))
for i, (model_name, predictions) in enumerate(models_predictions.items()):
    plt.subplot(3, 3, i + 1)
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
results = []
for name, clf in models_predictions.items():
    acc = accuracy_score(y_test, clf)
    f1 = f1_score(y_test, clf)
    precision = precision_score(y_test, clf)
    recall = recall_score(y_test, clf)
    results.append([name, acc, precision, recall, f1])

In [ ]:
#  Create a DataFrame for easy plotting
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"])
results_df = results_df.sort_values(by="Accuracy", ascending=False)
print("\nModel Performance Comparison:\n")
print(results_df)

In [ ]:
# PLOT 1: ACCURACY COMPARISON
# ============================
plt.figure(figsize=(10,6))
sns.barplot(x="Accuracy", y="Model", data=results_df, palette="viridis")
plt.title("Model Accuracy Comparison", fontsize=14)
plt.xlabel("Accuracy")
plt.ylabel("Model Name")
plt.xlim(0.8, 1.0)
plt.show()

In [ ]:
# PLOT 2: F1-SCORE COMPARISON
# ============================
plt.figure(figsize=(10,6))
sns.barplot(x="F1-Score", y="Model", data=results_df, palette="mako")
plt.title("Model F1-Score Comparison", fontsize=14)
plt.xlabel("F1-Score")
plt.ylabel("Model Name")
plt.xlim(0.8, 1.0)
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import pandas as pd

# Compute all metrics for comparison
comparison_results = []

for name, preds in models_predictions.items():
    acc = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    comparison_results.append([name, acc, precision, recall, f1])

# Convert to DataFrame
comparison_df = pd.DataFrame(comparison_results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"])
comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

# Display neatly
print("\n========== MODEL PERFORMANCE COMPARISON ==========")
print(comparison_df.to_string(index=False))
print("==================================================")

# ============================
# ROC CURVE COMPARISON (line plots only)
# ============================
plt.figure(figsize=(8,6))
for name, clf in classifiers.items():
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc:.2f})")

# Add Voting Classifier as well
if hasattr(voting_classifier, "predict_proba"):
    y_prob = voting_classifier.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"Voting Classifier (AUC={auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.title("ROC Curve Comparison", fontsize=14)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# ======================================================
# ðŸ“Š HEATMAP VISUALIZATION FOR PROJECT (Figure d)
# ======================================================

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

# Create a dictionary to store predictions and accuracy for each model
models_predictions = {name: clf.predict(X_test_scaled) for name, clf in classifiers.items()}
models_predictions['Voting Classifier'] = voting_predictions

# Compute confusion matrices and accuracies
conf_matrices = {}
accuracies = {}

for model_name, preds in models_predictions.items():
    cm = confusion_matrix(y_test, preds)
    acc = accuracy_score(y_test, preds)
    conf_matrices[model_name] = cm
    accuracies[model_name] = acc

# Display accuracy heatmap (overview performance)
plt.figure(figsize=(8, 4))
sns.heatmap(pd.DataFrame(accuracies, index=['Accuracy']), annot=True, cmap='coolwarm', fmt=".3f")
plt.title("Model Accuracy Comparison Heatmap (Figure d)")
plt.yticks(rotation=0)
plt.show()

# Display confusion matrix heatmaps for each classifier
plt.figure(figsize=(15, 10))
for i, (model_name, cm) in enumerate(conf_matrices.items()):
    plt.subplot(3, 3, i + 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', cbar=False)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.suptitle("Model Confusion Matrices ", fontsize=16, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================
# ðŸ“Š Figure 7: Confusion Matrix Heatmap (Final)
# =============================================
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# --- Use the best performing model (Voting Classifier) ---
y_pred = voting_classifier.predict(X_test_scaled)

# --- Compute normalized confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize by true labels

# --- Plot Heatmap ---
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor='white',
    cbar=True
)

plt.title("Figure 7. Confusion Matrix Heatmap for Pneumonia Detection Model", fontsize=12, pad=15)
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("True Class", fontsize=11)
plt.xticks(ticks=[0.5, 1.5], labels=['Normal', 'Pneumonia'], rotation=0)
plt.yticks(ticks=[0.5, 1.5], labels=['Normal', 'Pneumonia'], rotation=0)
plt.tight_layout()
plt.show()
